# Used Car Price Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `price`

## 2 · Project Overview

We predict **used car prices** based on brand, model year, mileage, fuel type, transmission, engine displacement, and ownership history.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a used car's brand, year, mileage, fuel type, transmission, engine size, and owner count, predict its market price.

## 5 · Why This Project Matters

- **Used car pricing** is a multi-billion dollar market.
- Accurate pricing helps buyers avoid overpaying.
- Dealers use algorithms for inventory acquisition.
- Demonstrates depreciation modeling with brand hierarchy.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 9,000 |
| **Features** | brand, year, mileage, fuel_type, transmission, engine_cc, owner_count |
| **Target** | price (continuous, USD) |
| **Range** | ~$1.5K – $120K |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "price"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: price
Save dir: E:\Github\Machine-Learning-Projects\Regression\Used Car Price Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 9,000 used car listings with vehicle features.

In [4]:
np.random.seed(SEED)
n = 9000
brand = np.random.choice(["Toyota", "Honda", "Ford", "BMW", "Mercedes", "Hyundai", "Kia"], n,
                          p=[0.2, 0.15, 0.15, 0.1, 0.1, 0.15, 0.15])
brand_base = {"Toyota": 25000, "Honda": 23000, "Ford": 28000, "BMW": 45000,
              "Mercedes": 50000, "Hyundai": 20000, "Kia": 19000}
base_price = np.array([brand_base[b] for b in brand], dtype=float)

year = np.random.randint(2005, 2025, n)
age = 2025 - year
mileage = np.round(age * np.random.uniform(8000, 18000, n) + np.random.normal(0, 5000, n), 0).clip(500, 300000).astype(int)
fuel_type = np.random.choice(["Petrol", "Diesel", "Hybrid", "Electric"], n, p=[0.4, 0.3, 0.15, 0.15])
transmission = np.random.choice(["Manual", "Automatic", "CVT"], n, p=[0.3, 0.5, 0.2])
engine_cc = np.random.choice([1000, 1200, 1500, 1800, 2000, 2500, 3000, 3500], n)
owner_count = np.random.choice([1, 2, 3, 4], n, p=[0.4, 0.35, 0.2, 0.05])

fuel_mult = np.where(fuel_type == "Electric", 1.15, np.where(fuel_type == "Hybrid", 1.08,
            np.where(fuel_type == "Diesel", 1.02, 1.0)))
trans_mult = np.where(transmission == "Automatic", 1.05, np.where(transmission == "CVT", 1.03, 1.0))

price = base_price * fuel_mult * trans_mult
price = price * (1 - 0.08 * age) * (1 - 0.000002 * mileage)
price = price + engine_cc * 2 - 3000 * (owner_count - 1)
price = np.round(price + np.random.normal(0, 2000, n), -2).clip(1500, 120000)

df = pd.DataFrame({"brand": brand, "year": year, "mileage": mileage, "fuel_type": fuel_type,
                    "transmission": transmission, "engine_cc": engine_cc,
                    "owner_count": owner_count, "price": price})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['price'].describe()}")
df.head()

Dataset shape: (9000, 8)
Target stats:
count     9000.000000
mean      9568.722222
std      10169.450018
min       1500.000000
25%       1500.000000
50%       5300.000000
75%      15200.000000
max      58400.000000
Name: price, dtype: float64


,brand,year,mileage,fuel_type,transmission,engine_cc,owner_count,price
0,Ford,2009,270704,Petrol,Manual,1500,3,1500.0
1,Kia,2022,35807,Petrol,Manual,1000,3,9200.0
2,Hyundai,2019,96133,Hybrid,CVT,1000,1,14100.0
3,BMW,2016,155566,Hybrid,Automatic,1000,3,3900.0
4,Toyota,2021,77847,Petrol,CVT,3000,1,16400.0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

DATA VALIDATION

Shape: (9000, 8)

Missing values:
brand           0
year            0
mileage         0
fuel_type       0
transmission    0
engine_cc       0
owner_count     0
price           0
dtype: int64

Duplicate rows: 13

Dtypes:
brand            object
year              int32
mileage           int64
fuel_type        object
transmission     object
engine_cc         int64
owner_count       int64
price           float64
dtype: object

Target 'price' confirmed.

Target stats:
count     9000.000000
mean      9568.722222
std      10169.450018
min       1500.000000
25%       1500.000000
50%       5300.000000
75%      15200.000000
max      58400.000000
Name: price, dtype: float64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["year"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Year"); axes[0][0].set_ylabel("Price"); axes[0][0].set_title("Year vs Price")

axes[0][1].scatter(df["mileage"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Mileage"); axes[0][1].set_ylabel("Price"); axes[0][1].set_title("Mileage vs Price")

df.boxplot(column=TARGET, by="brand", ax=axes[0][2])
axes[0][2].set_title("Price by Brand")
axes[0][2].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="fuel_type", ax=axes[1][0])
axes[1][0].set_title("Price by Fuel Type")

axes[1][1].scatter(df["engine_cc"], df[TARGET], alpha=0.2, s=10)
axes[1][1].set_xlabel("Engine CC"); axes[1][1].set_ylabel("Price"); axes[1][1].set_title("Engine vs Price")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=axes[1][2], square=True)
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `price`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Price ($)")

df.groupby("brand")[TARGET].median().sort_values().plot(kind="barh", ax=axes[1],
    color="steelblue", edgecolor="black")
axes[1].set_title("Median Price by Brand")
axes[1].set_xlabel("Price ($)")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")

Range: $1,500 to $58,400
Mean: $9,569, Median: $5,300


## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

Train: (7200, 7), Test: (1800, 7)
Target — Train mean: 9,515.62, Test mean: 9,781.11


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `brand`, `fuel_type`, `transmission` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Luxury brands create right skew — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["age"] = 2025 - X_train["year"]
X_test["age"] = 2025 - X_test["year"]

X_train["mileage_per_year"] = X_train["mileage"] / (X_train["age"] + 1)
X_test["mileage_per_year"] = X_test["mileage"] / (X_test["age"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['brand', 'year', 'mileage', 'fuel_type', 'transmission', 'engine_cc', 'owner_count', 'age', 'mileage_per_year']


## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

BASELINE: Linear Regression
RMSE : 5,707.10
MAE  : 3,945.17
R2   : 0.7047


## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Regressors:
                               Adjusted R-Squared  R-Squared         RMSE  Time Taken
Model                                                                                
CatBoostRegressor                        0.975329   0.975453  1645.530904    4.169354
LGBMRegressor                            0.974840   0.974966  1661.765908    0.090533
HistGradientBoostingRegressor            0.974613   0.974740  1669.231125    0.308081
XGBRegressor                             0.970940   0.971085  1785.923901    0.198042
RandomForestRegressor                    0.967247   0.967411  1896.014858    1.557569
GradientBoostingRegressor                0.966828   0.966994  1908.094538    1.043933
BaggingRegressor                         0.963898   0.964078  1990.595238    0.181053
ExtraTreesRegressor                      0.962614   0.962801  2025.681254    1.204022
DecisionTreeRegressor                    0.941299   0.941593  2538.256182    0.036394
ExtraTreeRegressor   

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

FLAML best model: catboost
RMSE: 1,647.06
R2  : 0.9754


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost RMSE: 1,628.71  (3.4s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[181]	valid_0's l2: 2.77267e+06
LightGBM RMSE: 1,665.13  (5.8s)


XGBoost failed: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by RMSE):
                      RMSE      MAE      R2
CatBoost           1628.71  1113.47  0.9760
FLAML              1647.06  1129.88  0.9754
LightGBM           1665.13  1121.89  0.9749
Linear Regression  5707.10  3945.17  0.7047

Top 3 models: ['CatBoost', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")


Detailed Metrics — Top 3:

  CatBoost:
    RMSE : 1,628.71
    MAE  : 1,113.47
    R2   : 0.9760

  FLAML:
    RMSE : 1,647.06
    MAE  : 1,129.88
    R2   : 0.9754

  LightGBM:
    RMSE : 1,665.13
    MAE  : 1,121.89
    R2   : 0.9749


## 24 · Error Analysis

Examine residuals from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

Residual stats (CatBoost):
  Mean: 18.46, Std: 1,628.60
  Min: -6,211.13, Max: 7,593.07
  Median: -21.76


## 25 · Interpretation and Business Insight

**Key findings:**
- **Brand** is the strongest price predictor (Mercedes/BMW vs. Kia/Hyundai).
- **Year** (age) has strong negative depreciation effect.
- **Mileage** independently reduces value.
- **Engine size** adds value (power/luxury proxy).
- **Owner count** penalizes price (~$3K per additional owner).

**Business takeaway:** Price = Brand tier × (1 - depreciation) × condition adjustments. Low-mileage single-owner luxury cars hold value best.

## 26 · Limitations

1. Synthetic depreciation curve — real depreciation is non-linear.
2. No specific model (only brand).
3. Missing accident history and service records.
4. No regional price variation.
5. Simplified fuel type effects.

## 27 · How to Improve This Project

1. Add specific model within each brand.
2. Include accident and service history.
3. Use real listings (AutoTrader, Cars.com).
4. Add geographic pricing variation.
5. Model non-linear depreciation (exponential decay).

## 28 · Production Considerations

- Deploy for instant pricing on listing platforms.
- Output price confidence intervals.
- Update monthly with new market data.
- Monitor by brand/region for market shifts.
- Integrate with VIN decoder for exact specifications.

## 29 · Common Mistakes

1. Treating depreciation as linear (it's exponential).
2. Ignoring brand hierarchy in pricing models.
3. Not considering fuel type trends (EV premiums).
4. Using MSRP as a feature (leakage if from same time).
5. Not accounting for seasonal price fluctuations.

## 30 · Mini Challenge / Exercises

1. Log-transform `price` and compare R² improvement.
2. Remove `brand` — how much does RMSE increase?
3. Plot depreciation curves by brand.
4. Create `luxury = 1 if brand in [BMW, Mercedes]` and test.
5. Analyze price premium for Electric vs. Petrol across ages.

## 31 · Final Summary / Key Takeaways

1. **Brand** is the primary price determinant.
2. **Age** (year) drives depreciation — the #2 factor.
3. **Mileage** provides independent value adjustment.
4. **Engine size** and **fuel type** add secondary signals.
5. **Owner count** penalizes value linearly.
6. **Non-linear depreciation** modeling would improve accuracy.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Regression\Used Car Price Prediction\metrics.json
{
  "CatBoost": {
    "rmse": 1628.71,
    "mae": 1113.47,
    "r2": 0.976
  },
  "LightGBM": {
    "rmse": 1665.13,
    "mae": 1121.89,
    "r2": 0.9749
  },
  "Linear Regression": {
    "rmse": 5707.1,
    "mae": 3945.17,
    "r2": 0.7047
  },
  "FLAML": {
    "rmse": 1647.06,
    "mae": 1129.88,
    "r2": 0.9754
  }
}
